In [1]:
import sys
sys.path.append('../..')

from pathlib import Path
import os
from glob import glob

import cv2
from PIL import Image
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from models.baseline_cnn import LitCNN
from models.resnet_18 import LitResNet18
from models.dataset import ImageDataset

%load_ext autoreload
%autoreload 2

In [2]:
def filter_dataset(
        df,
        path_to_dataset,
        class_names_to_remove=["yeezy_slide"],
        bad_images_md_path="../2-exploration/bad_images.md"
):
    """

    """
    # Вычищаем Yeezy Slide, т.к. они не укладываются в наши представления о кроссовках
    df = df.query('sneaker_class != "yeezy_slide"')
    # Фильтруем битые картинки, рисунки и т.п.
    # Плохие картинки мы отсмотрели вручную. Список можно увидеть в файле bad_images
    with open(bad_images_md_path) as f:
        bad_images = f.read().strip().split("\n")

    bad_image_paths = [
        image[image.find("(") + 1 : -1] for image in bad_images if len(image) > 0
    ]  # В md указаны пути в формате ![name](path). Поэтому путь это все, что находится в круглых скобках ()

    bad_image_paths = [
        os.path.relpath(
            image, start=path_to_dataset
        )  # Тут заменил, надо заменить пути в bad_images.md
        for image in bad_image_paths
    ]

    print(f"Bad images len: {len(bad_image_paths)}")

    df = df[df["path"].apply(lambda x: x not in bad_image_paths)]
    
    print(f"Dataframe size: {df.shape[0]}")

    return df

In [3]:
# Изображения хранятся в директории так, что каждой модели кроссовок соответствует
# своя папка. Чтобы можно было рассчитывать статистики, мы собираем в датафрейм относительный
# путь до каждого из изображений, и рассчитываем агрегаты на основе этого датафрейма
from src.data.utils.eda_utils import directory_to_dataframe

path_to_dataset = Path("../../data/01_raw/sneakers-dataset")
df = directory_to_dataframe(path_to_dataset)
df.head()

,path,sneaker_class
0,reebok_classic_leather/0071.jpg,reebok_classic_leather
1,reebok_classic_leather/0065.jpg,reebok_classic_leather
2,reebok_classic_leather/0059.jpg,reebok_classic_leather
3,reebok_classic_leather/0058.jpg,reebok_classic_leather
4,reebok_classic_leather/0064.jpg,reebok_classic_leather


In [4]:
# Вычищаем Yeezy Slide, т.к. они не укладываются в наши представления о кроссовках
df = filter_dataset(
    df,
    path_to_dataset
)

df.head()

Bad images len: 12
Dataframe size: 5796


,path,sneaker_class
0,reebok_classic_leather/0071.jpg,reebok_classic_leather
1,reebok_classic_leather/0065.jpg,reebok_classic_leather
2,reebok_classic_leather/0059.jpg,reebok_classic_leather
3,reebok_classic_leather/0058.jpg,reebok_classic_leather
4,reebok_classic_leather/0064.jpg,reebok_classic_leather


In [5]:
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["sneaker_class"],
    random_state=42
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.25,
    stratify=train_df["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

,path,sneaker_class
448,converse_chuck_70_low/0086.jpg,converse_chuck_70_low
2870,nike_dunk_low/0038.jpg,nike_dunk_low
2514,vans_old_skool/0059.jpg,vans_old_skool
3664,adidas_stan_smith/0001.jpg,adidas_stan_smith
3230,nike_air_jordan_1_high/0009.jpg,nike_air_jordan_1_high


(3477, 2)

,path,sneaker_class
2225,new_balance_327/0028.jpg,new_balance_327
504,reebok_club_c_85/0070.jpg,reebok_club_c_85
3991,asics_gel-lyte_iii/0017.jpg,asics_gel-lyte_iii
2127,nike_dunk_high/0050.jpg,nike_dunk_high
49,reebok_classic_leather/0019.jpg,reebok_classic_leather


(1159, 2)

,path,sneaker_class
1984,adidas_forum_low/0026.jpg,adidas_forum_low
4479,nike_air_jordan_4/0078.jpg,nike_air_jordan_4
1633,yeezy_boost_350_v2/0018.jpg,yeezy_boost_350_v2
3120,adidas_superstar/0028.jpg,adidas_superstar
1379,adidas_forum_high/0091.jpg,adidas_forum_high


(1160, 2)

In [6]:
train_paths = train_df["path"].tolist()
val_paths   = val_df["path"].tolist()
test_paths  = test_df["path"].tolist()

train_labels = train_df["sneaker_class"].tolist()
val_labels   = val_df["sneaker_class"].tolist()
test_labels  = test_df["sneaker_class"].tolist()

In [7]:
train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

In [8]:
train_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=train_paths,
    labels=train_labels,
    augmenter=train_tfms
)

val_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=val_paths,
    labels=val_labels,
    augmenter=val_tfms
)

test_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=test_paths,
    labels=test_labels,
    augmenter=val_tfms
)

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

In [ ]:
# model = LitCNN(num_classes=df["sneaker_class"].nunique())
resnet_model = LitResNet18(num_classes=df["sneaker_class"].nunique())

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    devices=1,
    logger=pl.loggers.TensorBoardLogger("lightning_logs", name="cnn_baseline"),
    log_every_n_steps=1
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


In [11]:
# import warnings 
# warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# trainer.fit(model, train_loader, val_loader)

In [12]:
import warnings 
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

trainer.fit(resnet_model, train_loader, val_loader)


  | Name    | Type             | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | model   | ResNet           | 11.2 M | train | 0    
1 | loss_fn | CrossEntropyLoss | 0      | train | 0    
-------------------------------------------------------------
11.2 M    Trainable params
0         Non-trainable params
11.2 M    Total params
44.807    Total estimated model params size (MB)
69        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


In [23]:
pred_batches = trainer.predict(resnet_model, dataloaders=[test_loader])

Predicting: |          | 0/? [00:00<?, ?it/s]

In [24]:
y_pred = torch.cat(pred_batches).cpu().numpy()

In [39]:
y_true = [x[1].numpy().item() for x in test_dataset]

In [42]:
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.79      0.63      0.70        30
           1       1.00      0.56      0.71        18
           2       0.42      0.93      0.58        30
           3       0.86      0.63      0.73        19
           4       1.00      0.29      0.44        14
           5       0.82      0.48      0.61        29
           6       1.00      0.32      0.48        19
           7       0.60      0.87      0.71        30
           8       0.80      0.67      0.73        18
           9       0.67      0.27      0.38        15
          10       0.75      0.50      0.60        30
          11       0.48      0.80      0.60        15
          12       0.67      0.11      0.19        18
          13       0.96      0.83      0.89        30
          14       0.75      0.68      0.71        22
          15       0.83      0.80      0.81        30
          16       0.59      0.77      0.67        30
          17       0.33    